In [ ]:
# configuration class for mouseReMoCo application

from enum import Enum
import copy

class TaskType(Enum):
    """Task types supported by the application"""

    CIRCULAR = "circular"
    LINEAR = "linear"

class OutputConfiguration:
    """Configuration for data output coordinate system (CSV, LSL).
    
    Creates a copy of the Configuration object and modifies it for output.
    Handles coordinate transformation: screen → center-origin with y-reversed (matplotlib style).
    """
    
    def __init__(self, config):
        """Initialize with Configuration object and create a working copy.
        
        Args:
            config: Configuration instance to copy and modify for output
        """
        # Store original screen center for transformation calculations
        self.origin_x = config.center_x
        self.origin_y = config.center_y
        
        self.config = copy.deepcopy(config)
        
        # update the config copy to reflect output coordinate system
        self.config.center_x = self.config.center_x - self.origin_x
        self.config.center_y = self.config.center_y - self.origin_y
        self.config.corner_x = self.config.corner_x - self.origin_x
        self.config.corner_y = self.config.corner_y - self.origin_y
        self.config.origin_mode = "center"
        self.config.y_reversed = True
        
    
    def transform_coordinates(self, x: float, y: float) -> tuple[float, float]:
        """Convert screen coordinates to output coordinates."""
        x_out = x - self.origin_x   # Use origin_x, not center_x
        y_out = self.origin_y - y   # Use origin_y, not center_y
        return x_out, y_out
    

class Configuration:
    """
    Main configuration class mirroring Java Configuration.java
    Handles all application settings including:
    - Screen and window configuration
    - Circular and linear task parameters
    - Visual styling (colors, cursors, fonts)
    - Input and output settings

    INSTANTIATION: Must be created with NO arguments
        config = Configuration()

    CUSTOMIZATION: Set properties explicitly before setup
        config.cursor_radius = 20
        config.cycle_max_number = 4

    RUNTIME: Call setup.create_and_display() to populate
        - screen_width, screen_height
        - drawable_width, drawable_height
        - center_x, center_y
        - all derived values (internal_limit, external_limit, etc.)
    """

    def __init__(self):
        """Initialize Configuration with NO arguments - all values use defaults.

        To customize behavior:
        1. Create: config = Configuration()
        2. Modify defaults as needed before setup
        3. Call: setup.create_and_display() which populates runtime values

        This design ensures clarity: anything not explicitly set before
        measure_and_correct_dimensions() gets its runtime value there.
        """
        # ===== Window Configuration =====
        self._title = "Wacom Tablet Test"
        self._target_monitor = 2
        self._width = None
        self._height = None
        self._nb_cursor_radii_for_target_margin = 5
        self.software = "mouseReMoCo-Python"
        self.version = "2.0.0"

        # ===== Screen & Window Configuration (set during measure_and_correct_dimensions) =====
        self.screen_width = 0
        self.screen_height = 0
        self._drawable_width = 0
        self._drawable_height = 0
        self._frame_location_x = 0
        self._frame_location_y = 0
        self._frame_insets = {"top": 0, "bottom": 0, "left": 0, "right": 0}
        self._frame_undecorated = False
        self._used_screen_id = 0

        # ===== Circular Task Parameters (radii set during measure_and_correct_dimensions) =====
        self.task_string = "circular"
        self.center_x = 0
        self.center_y = 0
        self.corner_x = 0
        self.corner_y = 0
        self.external_radius = 150
        self.internal_radius = 80
        self.border_radius = 1
        self.circle_perimeter_mm = 0

        # ===== Circular task derived values (set during measure_and_correct_dimensions) =====
        self.task_radius = 0.0
        self.tolerance_px = 0
        self.index_of_difficulty = 0.0
        self.internal_limit = 0
        self.external_limit = 0

        # ===== Linear Task Parameters =====
        self.inter_line_distance_mm = 150
        self.line_height_mm = 100
        self.mm2px = 0.0

        # ===== Auditory Rhythm =====
        self.half_period = 2000

        # ===== Cursor Configuration =====
        self.cursor_radius = 16
        self.cursor_color_record = (255, 0, 0)  # RGB red
        r, g, b = self.cursor_color_record
        self.cursor_color_record_outside = (
            max(0, r // 2),
            max(0, g // 2),
            max(0, b // 2),
        )
        self.cursor_color_wait = (255, 255, 0)  # RGB yellow

        # ===== Visual Styling =====
        self.border_color = (255, 255, 255)  # RGB white
        self.background_color = (0, 0, 0)  # RGB black
        self.text_color = (255, 255, 255)  # RGB white

        # ===== Sequence Configuration =====
        self.auto_start = 3600  # seconds before auto start
        self.cycle_max_number = 6  # Move-Rest cycle number
        self.cycle_duration = 20  # seconds for a Move or Rest (half-cycle)
        self.is_target_hidden_during_pause = False

        # ===== Font Configuration =====
        self.font_size = 20
        self.font_family = "Courier"

        # ===== Flags =====
        self.is_with_lsl = False  # Lab Streaming Layer
        self.is_with_pause_target = False

        # ===== Trail Configuration =====
        self.trail_mode = "path_length"  # Active trail mode
        self.trail_length = None  # Trail length in pixels; None means 10×cursor_radius

        # ===== Output Configuration =====
        self.output_config = None  # Will be created after center_x, center_y are determined    

        # ===== Application State =====
        self.step = ""

        # Initialize derived values
        self._update_circular_task()

    def _update_circular_task(self):
        """Update circular task derived values"""
        if self.task_string == "circular":
            # Limits of the path
            self.internal_limit = self.internal_radius + self.cursor_radius
            self.external_limit = (
                self.external_radius - self.cursor_radius - self.border_radius
            )

            # ID in the steering law (Accot & Zhai 1999)
            self.task_radius = (self.internal_limit + self.external_limit) / 2.0
            self.tolerance_px = self.external_limit - self.internal_limit

            if self.tolerance_px > 0:
                self.index_of_difficulty = (
                    2.0 * 3.14159 * self.task_radius
                ) / self.tolerance_px

    def get_trail_length(self) -> int:
        """Get trail length for current mode, defaulting to 10×cursor_radius if not set"""
        if self.trail_length is not None:
            return self.trail_length
        return 10 * self.cursor_radius

    def set_index_of_difficulty(self, index_of_difficulty: float):
        """Set index of difficulty and adjust circle parameters"""
        if self.task_string != "circular":
            return

        # Calculate new tolerance width
        w = (3.14159 * self.external_limit) / (index_of_difficulty + 3.14159)
        wn = round(2 * w)

        # Update internal limit and radius
        self.internal_limit = self.external_limit - wn
        self.internal_radius = self.internal_limit - self.cursor_radius

        # Recalculate derived values
        self._update_circular_task()

    def set_circular_task(self):
        """Initialize circular task parameters"""
        self._update_circular_task()

    def set_linear_task(self):
        """Initialize linear task parameters"""
        # Linear task setup would go here
        pass

    def set_circle_perimeter(self, perimeter_mm: int, screen_resolution_ppi: float):
        """Set circle perimeter and adjust circle parameters accordingly"""
        if perimeter_mm <= 0 or screen_resolution_ppi <= 0:
            return

        # Convert mm to pixels
        self.circle_perimeter_mm = perimeter_mm
        perimeter_px = perimeter_mm * screen_resolution_ppi / 25.4  # 25.4 mm per inch

        # Calculate new radius and tolerance
        self.task_radius = perimeter_px / (2.0 * 3.14159)
        tolerance = perimeter_px / self.index_of_difficulty

        external_limit = self.task_radius + tolerance / 2.0
        internal_limit = self.task_radius - tolerance / 2.0

        external_radius = external_limit + self.cursor_radius + self.border_radius
        internal_radius = internal_limit - self.cursor_radius

        self.external_radius = round(external_radius)
        self.internal_radius = round(internal_radius)

        self.corner_x = self._drawable_width // 2 - self.external_radius
        self.corner_y = self._drawable_height // 2 - self.external_radius

        self._update_circular_task()

    def calculate_default_circle_radii(
        self, screen_width: int, screen_height: int
    ) -> tuple[int, int]:
        """Calculate circle radii based on screen dimensions and margin settings"""
        # NOTE: default margin is 5 times cursor radius
        # Calculate available space accounting for margins
        margin_px = self._nb_cursor_radii_for_target_margin * self.cursor_radius
        available_width = screen_width - 2 * margin_px
        available_height = screen_height - 2 * margin_px

        # Use smaller dimension to ensure circle fits
        max_diameter = min(available_width, available_height)

        if max_diameter <= 0:
            return self.external_radius, self.internal_radius

        # External radius is half the maximum diameter
        external_radius = max_diameter // 2

        # Internal radius is 60% of external radius (creates 40% wide tolerance band)
        internal_radius = int(external_radius * 0.6)

        return external_radius, internal_radius

    def set_center_x(self, center_x: int):
        """Set center X and update corner X accordingly"""
        self.center_x = center_x
        self.corner_x = center_x - self.external_radius

    def set_center_y(self, center_y: int):
        """Set center Y and update corner Y accordingly"""
        self.center_y = center_y
        self.corner_y = center_y - self.external_radius

    def set_corner_x(self, corner_x: int):
        """Set corner X and update center X accordingly"""
        self.corner_x = corner_x
        self.center_x = corner_x + self.external_radius

    def set_corner_y(self, corner_y: int):
        """Set corner Y and update center Y accordingly"""
        self.corner_y = corner_y
        self.center_y = corner_y + self.external_radius

    def to_string(self) -> str:
        """Generate configuration string representation"""
        parts = [
            f"software: {self.software}",
            f"title: {self._title}",
            f"targetMonitor: {self._target_monitor}",
            f"isWithLSL: {self.is_with_lsl}",
            f"isWithPauseTarget: {self.is_with_pause_target}",
            f"screenWidth: {self.screen_width}",
            f"screenHeight: {self.screen_height}",
            f"drawableWidth: {self._drawable_width}",
            f"drawableHeight: {self._drawable_height}",
            f"frameLocationX: {self._frame_location_x}",
            f"frameLocationY: {self._frame_location_y}",
            f"frameUndecorated: {self._frame_undecorated}",
            f"usedScreenId: {self._used_screen_id}",
            f"centerX: {self.center_x}",
            f"centerY: {self.center_y}",
            f"marginMultiplier: {self._nb_cursor_radii_for_target_margin}",
            f"task: {self.task_string}",
            f"autoStart: {self.auto_start}",
            f"cycleMaxNumber: {self.cycle_max_number}",
            f"cycleDuration: {self.cycle_duration}",
            f"halfPeriod: {self.half_period}",
            f"borderColor: {self.border_color}",
            f"backgroundColor: {self.background_color}",
            f"textColor: {self.text_color}",
            f"cursorRadius: {self.cursor_radius}",
            f"cursorColorRecord: {self.cursor_color_record}",
            f"cursorColorWait: {self.cursor_color_wait}",
            f"fontSize: {self.font_size}",
            f"fontFamily: {self.font_family}",
            f"trailMode: {self.trail_mode}",
            f"trailLength: {self.trail_length}",
        ]

        if self.task_string == "circular":
            parts.extend(
                [
                    f"cornerX: {self.corner_x}",
                    f"cornerY: {self.corner_y}",
                    f"externalRadius: {self.external_radius}",
                    f"internalRadius: {self.internal_radius}",
                    f"internalLimit: {self.internal_limit}",
                    f"externalLimit: {self.external_limit}",
                    f"borderRadius: {self.border_radius}",
                    f"circlePerimeterMm: {self.circle_perimeter_mm}",
                    f"indexOfDifficulty: {self.index_of_difficulty:.2f}",
                    f"taskRadius: {self.task_radius:.2f}",
                    f"taskTolerance: {self.tolerance_px}",
                ]
            )
        elif self.task_string == "linear":
            parts.extend(
                [
                    f"interLineDistanceMm: {self.inter_line_distance_mm}",
                    f"lineHeightMm: {self.line_height_mm}",
                    f"mm2px: {self.mm2px:.2f}",
                ]
            )

        return "\n".join(parts)

In [ ]:
# Screen management — ScreenInfo + ScreenManager utilities
from dataclasses import dataclass

from PyQt6.QtWidgets import QApplication, QWidget


@dataclass
class ScreenInfo:
    """Information about a screen"""

    name: str
    index: int
    width: int
    height: int
    pos_x: int
    pos_y: int
    phys_width_mm: float
    phys_height_mm: float
    dpi_x: float
    dpi_y: float
    dpi_avg: float
    diag_inches: float


class ScreenManager:
    """Static utility methods for screen and window management"""

    @staticmethod
    def get_screen_info(screen, app: QApplication) -> ScreenInfo:
        """Extract detailed info from a QScreen object"""
        geometry = screen.geometry()
        phys_size = screen.physicalSize()

        # Calculate DPI
        dpi_x = geometry.width() / (phys_size.width() / 25.4)
        dpi_y = geometry.height() / (phys_size.height() / 25.4)
        dpi_avg = (dpi_x + dpi_y) / 2

        # Calculate diagonal in inches
        diag_inches = (phys_size.width() ** 2 + phys_size.height() ** 2) ** 0.5 / 25.4

        return ScreenInfo(
            name=screen.name(),
            index=app.screens().index(screen),
            width=geometry.width(),
            height=geometry.height(),
            pos_x=geometry.x(),
            pos_y=geometry.y(),
            phys_width_mm=phys_size.width(),
            phys_height_mm=phys_size.height(),
            dpi_x=dpi_x,
            dpi_y=dpi_y,
            dpi_avg=dpi_avg,
            diag_inches=diag_inches,
        )

    @staticmethod
    def get_all_screens(app: QApplication) -> list[ScreenInfo]:
        """Get info for all connected screens"""
        return [ScreenManager.get_screen_info(screen, app) for screen in app.screens()]

    @staticmethod
    def print_all_screens(screens: list[ScreenInfo]):
        """Print formatted screen information"""
        print("=" * 60)
        for s in screens:
            print(f"\nScreen {s.index + 1}: {s.name}")
            print(f"  Geometry: {s.width}×{s.height} @ ({s.pos_x}, {s.pos_y})")
            print(f"  DPI: {s.dpi_x:.1f}×{s.dpi_y:.1f} (avg: {s.dpi_avg:.1f})")
            print(f"  Physical: {s.phys_width_mm:.1f}×{s.phys_height_mm:.1f} mm")
            print(f'  Diagonal: {s.diag_inches:.1f}"')

    @staticmethod
    def get_target_screen(
        screens: list[ScreenInfo], config: Configuration
    ) -> ScreenInfo:
        """Get the target screen with safe fallback"""
        target_index = config._target_monitor - 1  # Convert 1-indexed to 0-indexed
        if 0 <= target_index < len(screens):
            return screens[target_index]
        print(f"⚠ Monitor {config._target_monitor} not found, using primary screen")
        return screens[0]

    @staticmethod
    def get_usable_screen_size(
        app: QApplication, screen_info: ScreenInfo
    ) -> tuple[int, int]:
        """Get usable screen size (excludes taskbars, etc.)"""
        screen = app.screens()[screen_info.index]
        usable = screen.availableGeometry()
        return usable.width(), usable.height()

    @staticmethod
    def get_window_drawable_area(
        widget: QWidget, initial_width: int, initial_height: int
    ) -> tuple[int, int, dict]:
        """Calculate actual drawable area accounting for window frame insets"""
        frame_geometry = widget.frameGeometry()
        content_geometry = widget.geometry()

        # Calculate frame insets
        insets = {
            "top": content_geometry.top() - frame_geometry.top(),
            "bottom": frame_geometry.bottom() - content_geometry.bottom(),
            "left": content_geometry.left() - frame_geometry.left(),
            "right": frame_geometry.right() - content_geometry.right(),
        }

        # Calculate actual drawable dimensions
        actual_width = initial_width - insets["left"] - insets["right"]
        actual_height = initial_height - insets["top"] - insets["bottom"]

        return actual_width, actual_height, insets


class TabletDetector:
    """Detect graphics tablets using Qt's QInputDevice"""

    @staticmethod
    def get_tablets():
        """Returns list of detected stylus/tablet devices"""
        from PyQt6.QtGui import QInputDevice

        tablets = []
        for device in QInputDevice.devices():
            if device.type() == QInputDevice.DeviceType.Stylus:
                tablets.append(device.name())
        return tablets

    @staticmethod
    def has_tablet():
        """Returns True if any tablet is detected"""
        return len(TabletDetector.get_tablets()) > 0

    @staticmethod
    def print_tablet_status():
        """Print tablet detection status to console"""
        tablets = TabletDetector.get_tablets()
        print("\n" + "=" * 60)
        if tablets:
            print(f"✓ Tablet detected: {tablets[0]}")
            if len(tablets) > 1:
                print(f"  ({len(tablets)} total devices found)")
        else:
            print("⚠ No tablet detected - will use mouse input only")
        print("=" * 60)

    @staticmethod
    def print_all_devices():
        """Debug: Print all input devices Qt sees"""
        from PyQt6.QtGui import QInputDevice

        print("\n" + "=" * 60)
        print("All detected input devices:")
        devices = QInputDevice.devices()
        if not devices:
            print("  (no devices found)")
        else:
            for device in devices:
                print(f"  - {device.name()}: {device.type()}")
        print("=" * 60)

In [ ]:
# Cursor Factory — Generate custom cursor images

from PyQt6.QtGui import QPixmap, QPainter, QColor, QCursor
from PyQt6.QtCore import Qt, QPoint


class CursorFactory:
    """Factory for creating custom cursor images with filled circles and crosshairs"""

    @staticmethod
    def create_cursor(
        radius: int,
        color: tuple[int, int, int],
        background_color: tuple[int, int, int] = (0, 0, 0),
    ) -> QCursor:
        """
        Create a custom cursor with a filled circle and center crosshair.

        Args:
            radius: Cursor circle radius in pixels
            color: RGB tuple (r, g, b) for circle color
            background_color: RGB tuple for background (for crosshair visibility)

        Returns:
            QCursor with the custom cursor image
        """
        diameter = radius * 2

        # Create transparent pixmap
        pixmap = QPixmap(diameter, diameter)
        pixmap.fill(Qt.GlobalColor.transparent)

        # Create painter and draw on pixmap
        painter = QPainter(pixmap)
        painter.setRenderHint(QPainter.RenderHint.Antialiasing)

        # Draw filled circle
        circle_color = QColor(*color)
        painter.setBrush(circle_color)
        painter.setPen(circle_color)
        painter.drawEllipse(0, 0, diameter, diameter)

        # Draw center crosshair (two perpendicular lines)
        crosshair_color = QColor(*background_color)
        painter.setPen(crosshair_color)

        crosshair_length = 4  # pixels extending from center in each direction
        center = radius

        # Horizontal line
        painter.drawLine(
            center - crosshair_length, center, center + crosshair_length, center
        )

        # Vertical line
        painter.drawLine(
            center, center - crosshair_length, center, center + crosshair_length
        )

        painter.end()

        # Create cursor with hotspot at center
        hotspot = QPoint(radius, radius)
        cursor = QCursor(pixmap, hotspot.x(), hotspot.y())

        return cursor

    @staticmethod
    def create_record_cursor(config: "Configuration") -> QCursor:
        """Create cursor for recording state (red circle)"""
        return CursorFactory.create_cursor(
            radius=config.cursor_radius,
            color=config.cursor_color_record,
            background_color=config.background_color,
        )

    @staticmethod
    def create_wait_cursor(config: "Configuration") -> QCursor:
        """Create cursor for waiting state (yellow circle)"""
        return CursorFactory.create_cursor(
            radius=config.cursor_radius,
            color=config.cursor_color_wait,
            background_color=config.background_color,
        )

    @staticmethod
    def create_out_cursor(config: "Configuration") -> QCursor:
        """Create cursor for outside target state (darkened record color)"""
        return CursorFactory.create_cursor(
            radius=config.cursor_radius,
            color=config.cursor_color_record_outside,
            background_color=config.background_color,
        )

In [ ]:
# Circular target rendering

from dataclasses import dataclass

from PyQt6.QtCore import Qt
from PyQt6.QtGui import QBrush, QColor, QPainter, QPen


@dataclass
class CircularTaskConfig:
    """Configuration for circular target task"""

    external_radius: int = 150  # pixels
    internal_radius: int = 80  # pixels
    background_color: str = "black"
    path_color: str = "#333333"  #  darkgray < "#333333"  < "#1a1a1a" < black
    circle_border_color: str = "white"
    circle_border_width: int = 2

    @staticmethod
    def rgb_to_hex(rgb_tuple: tuple[int, int, int]) -> str:
        """Convert RGB tuple (r, g, b) to hex color string"""
        r, g, b = rgb_tuple
        return f"#{r:02x}{g:02x}{b:02x}"


class CircularTargetWidget:
    """Draw circular target with tolerance band"""

    def __init__(self, config: "CircularTaskConfig | None" = None):
        self.config = config or CircularTaskConfig()

    def draw(self, painter: QPainter, center_x: int, center_y: int):
        """Draw the circular target"""
        painter.setRenderHint(QPainter.RenderHint.Antialiasing)

        # Draw external circle (border)
        self._draw_filled_circle(
            painter=painter,
            x=center_x,
            y=center_y,
            radius=self.config.external_radius,
            fill_color=self.config.path_color,
            border_color=self.config.circle_border_color,
            border_width=self.config.circle_border_width,
        )

        # Draw internal circle (background)
        self._draw_filled_circle(
            painter=painter,
            x=center_x,
            y=center_y,
            radius=self.config.internal_radius,
            fill_color=self.config.background_color,
            border_color=self.config.circle_border_color,
            border_width=self.config.circle_border_width,
        )

    def _draw_filled_circle(
        self,
        painter: QPainter,
        x: int,
        y: int,
        radius: int,
        fill_color: str,
        border_color: str,
        border_width: int,
    ):
        """Helper to draw filled circle with border"""
        # Set fill color
        fill = QColor(fill_color)
        painter.setBrush(QBrush(fill))

        # Set border (pen)
        border = QColor(border_color)
        pen = QPen(border)
        pen.setWidth(border_width)
        painter.setPen(pen)

        # Draw circle
        painter.drawEllipse(x - radius, y - radius, 2 * radius, 2 * radius)

In [ ]:
# AppStatus class — Centralized application and UI state management

import time


class AppStatus:
    """Centralized application and UI state management.

    Groups all application state logically:
    - Record/pause state
    - Input device state
    - Display/window state
    - Task/task execution state

    Extend this class as new features are added (LSL streaming, performance tracking, etc.)
    """

    def __init__(self):
        # ===== Recording & Output State =====
        self.is_recording = False  # default: not recording at start
        self.recording_start_time = None
        self.total_recording_duration = 0.0  # Accumulates across pause/resume cycles
        self.pause_count = 0  # How many times paused

        # ===== Input Device State =====
        self.tablet_detected = False
        self.active_input_type = "mouse"  # "mouse" or "tablet"
        self.last_input_time = None

        # ===== Display State =====
        self.fullscreen_mode = 0  # 0=windowed, 1=fullscreen
        self.show_debug_info = False  # Future: toggle debug rectangles

        # ===== Trail State =====
        self.current_trail_mode = "fixed"  # One of Trail.VALID_MODES

        # ===== Task State =====
        self.task_started = False
        self.task_start_time = None
        self.task_elapsed_time = 0.0

    def start_recording(self):
        """Start or resume recording"""
        self.is_recording = True
        self.recording_start_time = time.time()

    def pause_recording(self):
        """Pause recording and accumulate duration"""
        if self.is_recording and self.recording_start_time:
            elapsed = time.time() - self.recording_start_time
            self.total_recording_duration += elapsed
            self.pause_count += 1
        self.is_recording = False

    def toggle_recording(self):
        """Toggle recording state"""
        if self.is_recording:
            self.pause_recording()
        else:
            self.start_recording()
        return self.is_recording

    def get_status_string(self) -> str:
        """Get human-readable status for display or logging"""
        status_lines = [
            f"Recording: {'ON' if self.is_recording else 'PAUSED'}",
            f"Input: {self.active_input_type.upper()}",
            f"Trail Mode: {self.current_trail_mode.upper()}",
            f"Pause Count: {self.pause_count}",
        ]

        if self.task_started:
            status_lines.append(f"Task Time: {self.task_elapsed_time:.1f}s")

        return " | ".join(status_lines)

In [ ]:
# Window setup orchestration


class WindowSetup:
    """Encapsulates the complete window setup and initialization process.

    Manages both window AND configuration lifecycle, making WindowSetup
    the true orchestrator of the complete initialization workflow.

    Configuration is created internally and can be customized via the
    config property before calling create_and_display().
    """

    def __init__(
        self,
        app: QApplication,
        tablet_test_class: type,
    ):
        self.app = app
        self.tablet_test_class = tablet_test_class

        # Create configuration internally - owned by WindowSetup
        self.config = Configuration()

        self.screens = None
        self.target_screen_info: "ScreenInfo | None" = None
        self.usable_width: int = 0
        self.usable_height: int = 0
        self.widget = None
        self.app_status = AppStatus()

    def initialize_screens(self):
        """Step 1: Detect screens and select target"""
        self.screens = ScreenManager.get_all_screens(self.app)
        ScreenManager.print_all_screens(self.screens)
        self.target_screen_info = ScreenManager.get_target_screen(
            self.screens, self.config
        )
        self.usable_width, self.usable_height = ScreenManager.get_usable_screen_size(
            self.app, self.target_screen_info
        )

    def calculate_initial_radii(self) -> tuple[int, int, "CircularTaskConfig"]:
        """Step 2-3: Calculate initial radii and create circle config"""
        external_radius, internal_radius = self.config.calculate_default_circle_radii(
            screen_width=self.usable_width,
            screen_height=self.usable_height,
        )

        circle_config = CircularTaskConfig(
            external_radius=external_radius,
            internal_radius=internal_radius,
            background_color=CircularTaskConfig.rgb_to_hex(
                self.config.background_color
            ),
        )
        return external_radius, internal_radius, circle_config

    def create_widget(self) -> QWidget:
        """Step 4: Create and position window widget"""
        widget = self.tablet_test_class(
            self.target_screen_info,
            self.circle_config,
            self.usable_width,
            self.usable_height,
            config=self.config,
            window_setup=self,
            output_data=None,
            app_status=self.app_status,
        )
        widget.setWindowTitle(self.config._title)
        widget.move(self.target_screen_info.pos_x, self.target_screen_info.pos_y)

        window_width = self.config._width or self.usable_width
        window_height = self.config._height or self.usable_height
        widget.resize(window_width, window_height)

        return widget

    def measure_and_correct_dimensions(self):
        """Step 5-8: Measure frame insets and update widget with corrected dimensions"""
        if self.target_screen_info is None:
            raise RuntimeError(
                "initialize_screens() must be called before measure_and_correct_dimensions()"
            )
        # Show window to make frame insets calculable
        self.widget.show()
        self.app.processEvents()

        # Measure actual drawable area
        actual_width, actual_height, insets = ScreenManager.get_window_drawable_area(
            self.widget, self.usable_width, self.usable_height
        )
        print(
            f"\nWindow frame insets: Top={insets['top']}, Bottom={insets['bottom']}, Left={insets['left']}, Right={insets['right']}"
        )

        # Recalculate radii with actual drawable area
        external_radius, internal_radius = self.config.calculate_default_circle_radii(
            actual_width, actual_height
        )

        # Update config with corrected radii and actual dimensions
        self.config.screen_width = self.target_screen_info.width
        self.config.screen_height = self.target_screen_info.height
        self.config._drawable_width = actual_width
        self.config._drawable_height = actual_height
        self.config._frame_location_x = self.target_screen_info.pos_x
        self.config._frame_location_y = self.target_screen_info.pos_y
        self.config._frame_insets = insets
        self.config._used_screen_id = self.target_screen_info.index
        self.config.external_radius = external_radius
        self.config.internal_radius = internal_radius

        # Calculate and set center coordinates
        center_x = actual_width // 2
        center_y = actual_height // 2
        self.config.set_center_x(center_x)
        self.config.set_center_y(center_y)

        # Update derived values
        self.config._update_circular_task()

        # Create corrected circle config
        corrected_circle_config = CircularTaskConfig(
            external_radius=external_radius,
            internal_radius=internal_radius,
            background_color=CircularTaskConfig.rgb_to_hex(
                self.config.background_color
            ),
        )

        # Update widget with corrected values
        self.widget.circular_target = CircularTargetWidget(
            config=corrected_circle_config
        )
        self.widget.drawable_width = actual_width
        self.widget.drawable_height = actual_height
        self.widget.center_x = center_x
        self.widget.center_y = center_y
        self.widget.update()

    def finalize_display(self):
        """Step 9: Finalize window display and print configuration"""
        
        # Create OutputConfiguration after center is determined
        self.config.output_config = OutputConfiguration(self.config)
        
        # Create OutputData NOW = with the corrected configuration
        self.output_data = OutputTablet(
            config=self.config,
            app_status=self.app_status,
            output_config=self.config.output_config, 
            enable_csv=True,  # Enable CSV output
            enable_lsl=False,  # Set to True when LSL library available
        )

        # Assign output_data to widget and its trail
        self.widget.output_data = self.output_data
        self.widget.trail.output_data = self.output_data

        # Bring window to front and focus
        self.widget.raise_()
        self.widget.setFocus()

        # Display the final configuration
        print("\n" + "=" * 60)
        print(self.config.to_string())
        print("=" * 60 + "\n")

    def update_configuration(self, **kwargs):
        """Update configuration parameters and refresh the display.

        Args:
            **kwargs: Configuration parameters to update
                e.g., update_configuration(cursor_radius=20, index_of_difficulty=100)

        Supports any configuration property:
            - cursor_radius, cycle_max_number, background_color, etc.
            - index_of_difficulty (calls set_index_of_difficulty internally)
            - circle_perimeter (tuple: (perimeter_mm, screen_resolution_ppi))
        """
        for key, value in kwargs.items():
            if key == "index_of_difficulty":
                # Special handling for index_of_difficulty
                self.config.set_index_of_difficulty(value)
            elif key == "circle_perimeter":
                # Expects tuple: (perimeter_mm, screen_resolution_ppi)
                perimeter_mm, screen_resolution_ppi = value
                self.config.set_circle_perimeter(perimeter_mm, screen_resolution_ppi)
            elif hasattr(self.config, key):
                setattr(self.config, key, value)
            else:
                print(f"⚠ Warning: Configuration has no attribute '{key}'")

        # Update circular task derived values
        self.config._update_circular_task()

        # Recreate circle config with updated radii if needed
        corrected_circle_config = CircularTaskConfig(
            external_radius=self.config.external_radius,
            internal_radius=self.config.internal_radius,
            background_color=CircularTaskConfig.rgb_to_hex(
                self.config.background_color
            ),
        )

        # Update widget
        self.widget.circular_target = CircularTargetWidget(
            config=corrected_circle_config
        )
        self.widget.update()

    def create_and_display(self) -> QWidget:
        """Execute the complete setup pipeline"""
        self.initialize_screens()
        _, _, self.circle_config = self.calculate_initial_radii()
        self.widget = self.create_widget()
        self.measure_and_correct_dimensions()
        self.finalize_display()

In [ ]:
# Trail Class — Comet trail rendering and management

from collections import deque
import time

from PyQt6.QtGui import QColor, QPainter, QPen


class Trail:
    """
    Manages comet trail rendering.

    Modes:
    - path_length - fade based on cumulative distance traveled

    Trail data structure: deque of (x, y, timestamp, cumulative_distance, pressure)
    Each point remembers the pressure when it was recorded.
    """

    # Single source of truth for all valid trail modes
    VALID_MODES = ["path_length"]

    def __init__(self, config: "Configuration", app_status: "AppStatus | None" = None):
        """Initialize Trail with configuration reference and app status for tablet detection"""
        self.config = config
        self.app_status = app_status  # Reference to app status for tablet detection
        self.trail = deque()  # (x, y, timestamp, cumulative_path_distance, pressure)
        self.current_pressure = 0.0
        self.current_tilt_x = 0.0
        self.current_tilt_y = 0.0
        self.cumulative_distance = 0.0
        self._current_speed = 0.0
        self._last_x = None
        self._last_y = None
        self.output_data: "OutputTablet | None" = None
        # Set by MainWindow after OutputData is created

    def add_point(self, x: float, y: float, timestamp_ms: int, call_time_ms: int):
        """Add point with subpixel coordinates, pressure, and optional timestamps."""
        
        if self._last_x is not None and self._last_y is not None:
            dx = x - self._last_x
            dy = y - self._last_y
            distance = (dx**2 + dy**2) ** 0.5
            self._current_speed = distance
            self.cumulative_distance += distance
        else:
            self._current_speed = 0.0

        # Add point with timestamp, cumulative distance, and pressure
        self.trail.append(
            (x, y, time.time(), self.cumulative_distance, self.current_pressure)
        )

        # Update last position
        self._last_x = x
        self._last_y = y

        # Prune based on current mode
        self.prune()


    def prune(self):
        """Remove old points from trail based on path length"""
        # Path-distance-based pruning
        threshold = self.get_length()
        self.trail = deque(
            (x, y, t, d, p)
            for x, y, t, d, p in self.trail
            if self.cumulative_distance - d <= threshold
        )

    def get_length(self) -> float:
        """Compute trail length based on path_length mode"""
        base = self.config.get_trail_length()  # 10×cursor_radius or explicit value
        return base  # Uses cumulative distance, not Euclidean

    def _get_color_for_position(self, x: int, y: int) -> tuple[int, int, int]:
        """Determine trail color based on position relative to target"""
        dx = self.config.center_x - x
        dy = self.config.center_y - y
        distance = (dx * dx + dy * dy) ** 0.5

        is_inside = self.config.internal_limit < distance < self.config.external_limit

        # Mouse trails use blue, tablet trails use red
        # Use config colors based on position
        if is_inside:
            return self.config.cursor_color_record  # Inside target
        else:
            return self.config.cursor_color_record_outside  # Outside target

    def _draw_segment(
        self,
        painter: QPainter,
        x1: int,
        y1: int,
        x2: int,
        y2: int,
        pressure: float,
        opacity: float,
    ) -> bool:
        """Draw a single trail segment with conditional styling based on tablet detection.

        Returns True if segment was drawn, False if skipped (sub-pixel thickness).

        No tablet detected: 3px blue trail (fixed thickness, blue color)
        Tablet detected: Pressure-scaled red trail (thickness varies with pressure, position-based color)

        Both use opacity fading based on trail mode.
        """
        # Determine thickness and base color based on tablet detection
        if self.app_status and not self.app_status.tablet_detected:
            # NO TABLET → 3px BLUE trail (fixed thickness, blue color)
            thickness = 3
            base_color = (0, 0, 255)  # Blue RGB
        else:
            # TABLET DETECTED → Pressure-scaled RED trail (pressure-dependent thickness, position-based color)
            thickness = 2 * self.config.cursor_radius * pressure
            base_color = self._get_color_for_position(
                x2, y2
            )  # Red inside/dark red outside

        # Skip if thickness is too thin
        if thickness < 1:
            return False

        # Apply opacity fading to the color (same for both modes)
        color = QColor(*base_color)
        color.setAlpha(int(255 * opacity))  # Fade out as trail ages

        # Draw the segment
        painter.setPen(QPen(color, thickness))
        painter.drawLine(int(x1), int(y1), int(x2), int(y2))
        return True

    def draw(self, painter: QPainter, center_x: int, center_y: int):
        """Draw the entire trail with path-distance-based opacity fade"""
        if len(self.trail) < 2:
            return

        trail_list = list(self.trail)
        self._draw_path_distance_based(painter, trail_list)

    def _draw_path_distance_based(self, painter: QPainter, trail_list: list):
        """Draw trail with path-distance-based opacity fade"""
        threshold = self.get_length()
        for i in range(len(trail_list) - 1):
            x1, y1, _, d1, pressure1 = trail_list[i]
            x2, y2, _, d2, pressure2 = trail_list[i + 1]

            # Path distance from newest point
            path_distance = self.cumulative_distance - d2
            opacity = max(0, 1 - path_distance / threshold)

            self._draw_segment(painter, x1, y1, x2, y2, pressure2, opacity)

    def clear(self):
        """Clear all trail points and reset distance tracking"""
        self.trail.clear()
        self.cumulative_distance = 0.0
        self._current_speed = 0.0
        self._last_x = None
        self._last_y = None

    def set_mode(self, mode_name: str):
        """Switch to a new trail mode and clear trail"""

In [ ]:
# OutputTablet class -- Manage output of tablet/mouse to CSV files or to LSL stream

# Output Backend Architecture — With subpixel coordinate support

from abc import ABC, abstractmethod
import csv
from datetime import datetime


class OutputBackend(ABC):
    """Abstract base class for output targets (CSV, LSL, etc.)"""

    @abstractmethod
    def write_data(
        self,
        event_timestamp_ms: int,
        call_time_ms: int,
        x: float,
        y: float,
        is_inside: bool,
        pressure: float = 0.0,
        tilt_x: float = 0.0,
        tilt_y: float = 0.0,
    ):
        """Write position data point with subpixel precision.

        Args:
            event_timestamp_ms: Hardware event timestamp from Qt event
            call_time_ms: System time when event handler was called
        """
        pass

    @abstractmethod
    def write_marker(self, marker_text: str):
        """Write event marker"""
        pass

    @abstractmethod
    def close(self):
        """Close backend resources"""
        pass


class CSVBackend(OutputBackend):
    """CSV file output backend (data.csv, marker.csv) with subpixel support"""

    def __init__(self, config, output_config=None, data_filename="data.csv", marker_filename="marker.csv"):
        # Use output_config's modified copy if available, otherwise use original
        self.config = output_config.config if output_config else config
        self.output_config = output_config
        self.data_filename = data_filename
        self.marker_filename = marker_filename
        self.creation_timestamp = datetime.now()

        self.data_file = None
        self.marker_file = None
        self.data_writer = None
        self.marker_writer = None

        self._init_files()

    def _init_files(self):
        """Create and write headers to both CSV files"""
        # Create data.csv
        self.data_file = open(self.data_filename, "w", newline="")
        self.data_writer = csv.writer(self.data_file)

        # Write configuration header
        config_line = self._config_to_string()
        self.data_file.write(config_line + "\n")

        # Write creation timestamp as ISO8601 with timezone (for manual log inspection)
        timestamp_str = self.creation_timestamp.astimezone().isoformat()

        self.data_file.write(timestamp_str + "\n")
        self.data_file.write("\n")

        # Write column headers (updated to reflect subpixel precision)
        self.data_writer.writerow(
            [
                "event_timestamp",  # from event.timestamp()
                "call_time",  # from time.time() at handler call
                "mouseX",
                "mouseY",
                "mouseInTarget",
                "pressure",
                "tiltX",
                "tiltY",
            ]
        )
        self.data_file.flush()

        # Create marker.csv
        self.marker_file = open(self.marker_filename, "w", newline="")
        self.marker_writer = csv.writer(self.marker_file)

        # Write same headers to marker file
        self.marker_file.write(config_line + "\n")
        self.marker_file.write(timestamp_str + "\n")
        self.marker_file.write("\n")

        # Write marker column headers
        self.marker_writer.writerow(["timestamp", "milliseconds", "marker"])
        self.marker_file.flush()

        print(f"✓ CSV Backend: Created {self.data_filename} and {self.marker_filename}")

    def _config_to_string(self) -> str:
        """Convert all configuration attributes to semicolon-separated string for CSV header"""
        config_dict = {}
        
        # Write all config attributes
        for key, value in self.config.__dict__.items():
            # Skip private/protected attributes
            if key.startswith('_'):
                continue
            
            # Format the value appropriately
            if isinstance(value, bool):
                config_dict[key] = str(value).lower()
            elif isinstance(value, float):
                config_dict[key] = round(value, 2)
            else:
                config_dict[key] = value
        
        return ";".join([f"{k} {v}" for k, v in config_dict.items()])
        
        
    def write_data(
        self,
        event_timestamp_ms: int,
        call_time_ms: int,
        x: float,
        y: float,
        is_inside: bool,
        pressure: float = 0.0,
        tilt_x: float = 0.0,
        tilt_y: float = 0.0,
    ):
        """Write event data to CSV with coordinate transformation"""
        try:
            # Transform coordinates if output_config is available
            if self.output_config:
                x, y = self.output_config.transform_coordinates(x, y)
                
            self.data_writer.writerow(
                [
                    event_timestamp_ms,
                    call_time_ms,
                    round(x, 2),  # Subpixel precision: 2 decimal places
                    round(y, 2),  # Subpixel precision: 2 decimal places
                    1 if is_inside else 0,
                    round(pressure, 3),
                    round(tilt_x, 2),
                    round(tilt_y, 2),
                ]
            )
            self.data_file.flush()
        except Exception as e:
            print(f"ERROR writing data to {self.data_filename}: {e}")

    def write_marker(self, marker_text: str):
        """Write event marker to CSV"""
        try:
            current_time = datetime.now()
            # millisecond accuracy for humans and machines
            timestamp_ms = int(current_time.timestamp() * 1000)
            timestamp_str = current_time.strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]

            self.marker_writer.writerow(
                [
                    timestamp_str,  # Human-readable for manual log inspection
                    timestamp_ms,  # Epoch ms for automated sync
                    marker_text,
                ]
            )
            self.marker_file.flush()
        except Exception as e:
            print(f"ERROR writing marker to {self.marker_filename}: {e}")

    def close(self):
        """Close CSV files"""
        try:
            if self.data_file:
                self.data_file.close()
                print(f"✓ Closed {self.data_filename}")
            if self.marker_file:
                self.marker_file.close()
                print(f"✓ Closed {self.marker_filename}")
        except Exception as e:
            print(f"ERROR closing CSV files: {e}")


class LSLBackend(OutputBackend):
    """LSL (Lab Streaming Layer) output backend with subpixel support"""

    def __init__(self, config):
        self.config = config
        self.data_outlet = None
        self.marker_outlet = None
        self.numeric_marker_outlet = None

        try:
            import lsl as lsl_module

            self.lsl = lsl_module
            self._init_lsl()
            print("✓ LSL Backend: Initialized successfully")
        except ImportError:
            print("⚠ LSL library not available - LSL backend disabled")
            self.lsl = None

    def _init_lsl(self):
        """Initialize LSL streams for data and markers"""
        if not self.lsl:
            return

        # Data stream (float32 supports subpixel precision)
        data_info = self.lsl.StreamInfo(
            name="MouseData",
            type="MoCap",
            channel_count=3,
            nominal_srate=self.lsl.IRREGULAR_RATE,
            channel_format=self.lsl.cf_float32,
            source_id="mouseReMoCo",
        )

        # Add channel descriptions
        chns = data_info.desc().append_child("channels")
        labels = ["mouseX", "mouseY", "mouseInTarget"]
        types = ["PositionX", "PositionY", "flag"]
        units = ["pixels", "pixels", "boolean"]

        for label, type_, unit in zip(labels, types, units):
            chns.append_child("channel").append_child_value(
                "label", label
            ).append_child_value("type", type_).append_child_value("unit", unit)

        self.data_outlet = self.lsl.StreamOutlet(data_info)

        # Marker stream
        marker_info = self.lsl.StreamInfo(
            name="MouseMarkers",
            type="Markers",
            channel_count=1,
            nominal_srate=self.lsl.IRREGULAR_RATE,
            channel_format=self.lsl.cf_string,
            source_id="mouseReMoCo_markers",
        )
        self.marker_outlet = self.lsl.StreamOutlet(marker_info)

        # Numeric marker stream (for sync)
        numeric_info = self.lsl.StreamInfo(
            name="MouseMarkersNumeric",
            type="Markers",
            channel_count=1,
            nominal_srate=self.lsl.IRREGULAR_RATE,
            channel_format=self.lsl.cf_int32,
            source_id="mouseReMoCo_markers_numeric",
        )
        self.numeric_marker_outlet = self.lsl.StreamOutlet(numeric_info)

    def write_data(
        self,
        event_timestamp_ms: int,
        call_time_ms: int,
        x: float,
        y: float,
        is_inside: bool,
        pressure: float = 0.0,
        tilt_x: float = 0.0,
        tilt_y: float = 0.0,
    ):
        """Push position data to LSL with full subpixel precision"""
        if not self.lsl or not self.data_outlet:
            return

        try:
            # LSL float32 preserves subpixel precision
            sample = [float(x), float(y), float(1 if is_inside else 0)]
            self.data_outlet.push_sample(sample)
        except Exception as e:
            print(f"ERROR pushing data to LSL: {e}")

    def write_marker(self, marker_text: str):
        """Push event marker to LSL"""
        if not self.lsl or not self.marker_outlet:
            return

        try:
            sample = [marker_text]
            self.marker_outlet.push_sample(sample)
        except Exception as e:
            print(f"ERROR pushing marker to LSL: {e}")

    def close(self):
        """Close LSL outlets"""
        if not self.lsl:
            return

        try:
            if self.data_outlet:
                self.data_outlet = None
            if self.marker_outlet:
                self.marker_outlet = None
            if self.numeric_marker_outlet:
                self.numeric_marker_outlet = None
            print("✓ Closed LSL Backend")
        except Exception as e:
            print(f"ERROR closing LSL: {e}")


class OutputTablet:
    """Unified output manager with pluggable backends (CSV, LSL) - Subpixel support"""

    def __init__(self, config, app_status, output_config=None, enable_csv: bool = True, enable_lsl: bool = False):
        """
        Initialize OutputTablet with desired backends.

        Args:
            config: Configuration object with screen and task parameters
            app_status: AppStatus instance for recording state tracking
            output_config: OutputConfiguration instance for coordinate transformation (default: None)
            enable_csv: Enable CSV file output with subpixel precision (default: True)
            enable_lsl: Enable LSL streaming output (default: False)
        """
        self.config = config
        self.app_status = app_status  # For state tracking if needed
        self.backends = []

        if enable_csv:
            self.backends.append(CSVBackend(config, output_config))

        if enable_lsl:
            lsl_backend = LSLBackend(config)
            if lsl_backend.lsl:  # Only add if LSL initialized successfully
                self.backends.append(lsl_backend)
            else:
                print("⚠ LSL Backend not added due to initialization failure")

        if not self.backends:
            raise ValueError("At least one backend must be enabled (CSV or LSL)")

    def write_data(
        self,
        event_timestamp_ms: int,
        call_time_ms: int,
        x: float,
        y: float,
        is_inside: bool,
        pressure: float = 0.0,
        tilt_x: float = 0.0,
        tilt_y: float = 0.0,
    ):
        """Write position data with subpixel precision to all active backends"""
        # Only write if recording
        if not self.app_status.is_recording:
            return

        for backend in self.backends:
            backend.write_data(
                event_timestamp_ms,
                call_time_ms,
                x,
                y,
                is_inside,
                pressure,
                tilt_x,
                tilt_y,
            )

    def write_marker(self, marker_text: str):
        """Write event marker to all active backends"""
        for backend in self.backends:
            backend.write_marker(marker_text)

    def close(self):
        """Close all backends gracefully"""
        for backend in self.backends:
            backend.close()

    def __del__(self):
        """Ensure backends are closed when object is garbage collected"""
        self.close()

In [ ]:
# DataCapture class — Event data extraction and normalization

import time


class DataCapture:
    """Centralized event data extraction and normalization.
    
    Captures event data once and provides normalized output for:
    - CSV persistence (OutputCSV)
    - Trail rendering (Trail)
    - Cursor updates
    
    Key principle: Compute expensive operations (circle detection) 
    once per event, not multiple times.
    """
    
    def __init__(self, config: "Configuration"):
        """Initialize with configuration for circle detection."""
        self.config = config

    def capture_tablet_event(self, event) -> dict:
        """Extract and normalize tablet event data.
        
        Returns dict with:
        - x, y: subpixel coordinates
        - pressure: 0.0-1.0
        - event_timestamp_ms: native tablet event time
        - computed_timestamp_ms: system time
        - is_inside: whether inside target circle
        - input_type: "tablet"
        """
        x = event.position().x()
        y = event.position().y()
        pressure = event.pressure()
        # event_timestamp_ms = event.timestamp()
        event_timestamp_ms = int(event.timestamp())  
        computed_timestamp_ms = int(time.time() * 1000)
        is_inside = self._is_point_inside_circle(x, y)
        
        return {
            "x": x,
            "y": y,
            "pressure": pressure,
            "event_timestamp_ms": event_timestamp_ms,
            "computed_timestamp_ms": computed_timestamp_ms,
            "is_inside": is_inside,
            "input_type": "tablet",
        }

    def capture_mouse_event(self, event) -> dict:
        """Extract and normalize mouse event data.
        
        Returns dict (same format as capture_tablet_event, pressure=0.0)
        """
        x = event.position().x()
        y = event.position().y()
        pressure = 0.0
        # event_timestamp_ms = event.timestamp()
        event_timestamp_ms = int(event.timestamp())  
        computed_timestamp_ms = int(time.time() * 1000)
        is_inside = self._is_point_inside_circle(x, y)
        
        return {
            "x": x,
            "y": y,
            "pressure": pressure,
            "event_timestamp_ms": event_timestamp_ms,
            "computed_timestamp_ms": computed_timestamp_ms,
            "is_inside": is_inside,
            "input_type": "mouse",
        }

    def _is_point_inside_circle(self, x: float, y: float) -> bool:
        """Calculate if point is inside target circle."""
        dx = self.config.center_x - x
        dy = self.config.center_y - y
        distance = (dx * dx + dy * dy) ** 0.5
        return (
            self.config.internal_limit < distance < self.config.external_limit
        )

In [ ]:
# MainWindow - Main application window

import sys

from PyQt6.QtCore import Qt
from PyQt6.QtGui import QColor, QPainter, QPen, QTabletEvent
from PyQt6.QtWidgets import QApplication, QWidget


class MainWindow(QWidget):
    """Main application window handling rendering and input events."""

    # Map keyboard keys to trail modes from Trail class
    # Dynamically built to stay in sync if Trail.VALID_MODES changes
    TRAIL_MODES = {str(i + 1): mode for i, mode in enumerate(Trail.VALID_MODES)}

    # Command dispatch mapping for keyboard shortcuts
    KEY_COMMANDS = {
        'q': '_quit_application',
        'c': '_print_config',
        ' ': '_toggle_recording',
        'f': '_toggle_fullscreen',
    }


    def __init__(
        self,
        screen_info,
        circle_config,
        usable_width: int,
        usable_height: int,
        actual_drawable_width: int | None = None,
        actual_drawable_height: int | None = None,
        config: "Configuration | None" = None,
        window_setup: "WindowSetup | None" = None,
        output_data: "OutputTablet | None" = None,
        app_status: "AppStatus | None" = None,
    ):
        super().__init__()
        self.screen_info = screen_info
        self.config = config
        self.window_setup = window_setup
        self.data_capture = DataCapture(config) if config else None
        self.circular_target = CircularTargetWidget(config=circle_config)
        self.usable_width = usable_width
        self.usable_height = usable_height
        # Use actual drawable dimensions if provided, otherwise use usable dimensions
        self.drawable_width = actual_drawable_width or usable_width
        self.drawable_height = actual_drawable_height or usable_height
        self.center_x = usable_width // 2
        self.center_y = usable_height // 2

        # Single trail with tablet detection awareness
        self.trail = Trail(
            config, app_status=app_status
        )  
        self.last_mouse_x = None
        self.last_mouse_y = None

        # Accept OutputData passed from WindowSetup (created after config correction)
        self.output_data = output_data
        if self.output_data:
            self.trail.output_data = self.output_data

        # Use shared AppStatus instance from WindowSetup
        self.status = app_status if app_status else AppStatus()

        # Enable mouse tracking to receive mouseMoveEvent even when no button is pressed
        self.setMouseTracking(True)
        self.setFocus()

        # Print controls on startup
        self._print_control_instructions() 

    def paintEvent(self, event):
        """Handle all drawing operations"""
        painter = QPainter(self)
        # Use background color from config, or default to black
        if self.config and self.config.background_color:
            bg_color = QColor(*self.config.background_color)
        else:
            bg_color = Qt.GlobalColor.black
        painter.fillRect(self.rect(), bg_color)

        # Draw the circular target
        self.circular_target.draw(painter, self.center_x, self.center_y)

        # Draw the unified trail
        self.trail.draw(painter, self.center_x, self.center_y)

        # Draw mode indicator on screen
        self._draw_mode_indicator(painter)

        # Optionally draw limits rectangles for debugging
        self._draw_limits_retangles(painter)

    def _draw_limits_retangles(self, painter: QPainter):
        """Draw non-filled rectangles showing drawable screen limits"""
        # Draw green-yellow rectangles showing drawable screen limits

        original_brush = painter.brush()
        painter.setBrush(Qt.BrushStyle.NoBrush)

        shift = 0  # small shift to see the border more clearly (-1,suppresses the green rect)
        painter.setPen(QPen(Qt.GlobalColor.green, 1))
        painter.drawRect(
            shift,
            shift + 1,  # drawRect needs this correction (test on OSx)
            self.drawable_width - 2 * shift,
            self.drawable_height - 2 * shift - 1,
        )
        shift += 5
        painter.setPen(QPen(Qt.GlobalColor.yellow, 1))
        painter.drawRect(
            shift,
            shift + 1,
            self.drawable_width - 2 * (shift),
            self.drawable_height - 2 * (shift) - 1,
        )
        painter.setBrush(original_brush)

    def _draw_mode_indicator(self, painter: QPainter):
        """Draw current trail mode and recording status in corner"""
        mode_text = f"Mode: {self.config.trail_mode.upper()}"
        record_text = "● RECORDING" if self.status.is_recording else "○ PAUSED"

        # Draw mode (white)
        painter.setPen(QPen(Qt.GlobalColor.white))
        painter.drawText(10, 20, mode_text)

        # Draw recording status (green if ON, red if PAUSED)
        status_color = (
            Qt.GlobalColor.green if self.status.is_recording else Qt.GlobalColor.red
        )
        painter.setPen(QPen(status_color))
        painter.drawText(10, 40, record_text)


    def _toggle_recording(self):
        """Toggle recording on/off with spacebar"""
        self.status.toggle_recording()

        # Write marker to CSV
        marker = "RecordingStarted" if self.status.is_recording else "RecordingPaused"
        if self.output_data:
            self.output_data.write_marker(marker)

        # Print status to console (use unified method)
        self._print_status(self.status.get_status_string())

        # Redraw screen
        self.update()


    def _print_config(self):
        """Print configuration to console"""
        print("\n" + "=" * 60)
        print(self.config.to_string())
        print("=" * 60 + "\n")


    def _print_status(self, title: str, message: str = ""):
        """Print formatted status message with consistent formatting.
        
        Centralizes console output for status updates, making it easy to:
        - Redirect output to logs or UI
        - Change formatting globally
        - Suppress output (e.g., in tests)
        
        Args:
            title: Main message/title to display
            message: Optional secondary message (printed on new line if provided)
        """
        print("\n" + "=" * 60)
        print(title)
        if message:
            print(message)
        print("=" * 60 + "\n")


    def _print_fullscreen_status(self, mode: str):
        """Print fullscreen mode change notification.
        
        Args:
            mode: Either "BORDERLESS" or "WINDOWED"
        """
        if mode.upper() == "BORDERLESS":
            self._print_status("✓ Fullscreen Mode Changed", 
                             "Switched to BORDERLESS FULLSCREEN (no system UI access)")
        else:
            self._print_status("✓ Fullscreen Mode Changed", 
                             "Switched to WINDOWED mode")
            

    def _toggle_fullscreen(self):
        """Toggle between windowed and borderless fullscreen"""
        if self.status.fullscreen_mode == 0:
            # Switch to borderless fullscreen (no system UI)
            self.setWindowFlags(Qt.WindowType.FramelessWindowHint)
            self.setGeometry(self.screen().geometry())
            self.showFullScreen()
            self.status.fullscreen_mode = 1
            self._print_fullscreen_status("BORDERLESS")
        else:
            # Return to windowed
            self.setWindowFlags(Qt.WindowType.Widget)
            self.showNormal()
            self.status.fullscreen_mode = 0
            self._print_fullscreen_status("WINDOWED")

        # Process events to apply window state changes
        QApplication.instance().processEvents()

        # Recalculate drawable dimensions using WindowSetup's method
        if self.window_setup:
            self.window_setup.measure_and_correct_dimensions()

        self.setFocus()

    def _print_tablet_detected(self):
        """Print tablet detection confirmation"""
        self._print_status("✓ Tablet detected and active!")

    def _print_control_instructions(self):
        """Print keyboard control instructions on startup"""
        print(
            f"\n{'='*60}\nControls:\n"
            f"Press F: Toggle Fullscreen (Windowed ↔ Borderless)\n"
            f"Press C: Print Configuration\n"
            f"Press Q: Quit\n"
            f"Press SPACE: Toggle Record/Pause\n"
            f"{'='*60}\n"
        )

    def _quit_application(self):
        """Gracefully quit application, handling fullscreen state"""
        # Disable all input to suppress user interaction during shutdown
        self.setEnabled(False)

        # If in fullscreen, toggle to windowed first, wait 1 sec, then close
        # (fullscreen close is buggy on macOS, so bypass by going windowed first)
        if self.status.fullscreen_mode == 1:
            self._toggle_fullscreen()
            # Delay close by 1 second to let window state settle
            from PyQt6.QtCore import QTimer

            QTimer.singleShot(1000, self.close)
        else:
            # Already windowed, close immediately
            self.close()

    def closeEvent(self, event):
        """Handle window close - exit fullscreen before closing"""
        # If in fullscreen, explicitly return to windowed BEFORE closing
        if self.status.fullscreen_mode == 1:
            self.setWindowFlags(Qt.WindowType.Widget)
            self.showNormal()
            self.status.fullscreen_mode = 0
            # CRITICAL: Let windowing system process state changes before accepting close
            QApplication.instance().processEvents()

        # Stop recording and log end marker before closing
        if self.status.is_recording:
            self.status.pause_recording()
            if self.output_data:
                self.output_data.write_marker("RecordingEnded")

        # Close output data files before exiting
        if self.output_data:
            self.output_data.close()
        event.accept()

    def showEvent(self, event):
        """Initialize cursor when widget is shown"""
        super().showEvent(event)
        # Set initial cursor to "out" state
        self.setCursor(CursorFactory.create_out_cursor(self.config))

    def _update_cursor_for_position(self, x: float, y: float):
        """Update cursor color based on distance from circle center"""
        dx = self.center_x - x
        dy = self.center_y - y
        distance = (dx * dx + dy * dy) ** 0.5

        is_inside = self.config.internal_limit < distance < self.config.external_limit

        if is_inside:
            self.setCursor(CursorFactory.create_record_cursor(self.config))
        else:
            self.setCursor(CursorFactory.create_out_cursor(self.config))

    def _process_input_event(self, data: dict):
        """Unified input processing for both mouse and tablet events.
        
        Handles all common operations for any input type:
        - Write data to output backends (CSV, LSL)
        - Update trail with position and timestamp
        - Update cursor appearance based on position
        - Trigger screen repaint
        
        Args:
            data: Dictionary from DataCapture.capture_*_event() containing:
                  x, y, pressure, event_timestamp_ms, computed_timestamp_ms, is_inside
        """
        # Write to CSV/LSL output
        if self.output_data:
            self.output_data.write_data(
                event_timestamp_ms=data["event_timestamp_ms"],
                call_time_ms=data["computed_timestamp_ms"],
                x=data["x"],
                y=data["y"],
                is_inside=data["is_inside"],
                pressure=data["pressure"],
                tilt_x=0.0,
                tilt_y=0.0,
            )
        
        # Update trail with captured data
        self.trail.current_pressure = data["pressure"]
        self.trail.add_point(
            data["x"],
            data["y"],
            timestamp_ms=data["event_timestamp_ms"],
            call_time_ms=data["computed_timestamp_ms"],
        )
        
        # Update cursor based on position
        self._update_cursor_for_position(data["x"], data["y"])
        
        # Trigger repaint
        self.update()

    def _detect_tablet(self, data: dict):
        """Detect and log first tablet input with pressure."""
        if not self.status.tablet_detected and data["pressure"] > 0:
            self.status.tablet_detected = True
            self.status.active_input_type = "tablet"
            if self.output_data:
                self.output_data.write_marker("TabletDetected")
            self._print_tablet_detected()  

    def tabletOrMouseEvent(self, data: dict, is_tablet: bool = False):
        """Orchestrate input event handling for tablet or mouse.
        
        Central handler that routes input-specific logic while delegating
        common processing to _process_input_event().
        
        Args:
            data: Dictionary from DataCapture.capture_*_event()
            is_tablet: True for tablet events, False for mouse events
        """
        # Tablet-specific: detect on first input
        if is_tablet:
            self._detect_tablet(data)
        
        # Common processing for both input types
        self._process_input_event(data)
        
        # Mouse-specific: store position for reference
        if not is_tablet:
            self.last_mouse_x = data["x"]
            self.last_mouse_y = data["y"]

    def tabletEvent(self, event: QTabletEvent):
        """Handle tablet input event"""
        # Capture data
        data = self.data_capture.capture_tablet_event(event)
        
        # Process (detect tablet, write output, update trail/cursor/display)
        self.tabletOrMouseEvent(data, is_tablet=True)
        
        # Mark event as handled
        event.accept()

    def mouseMoveEvent(self, event):
        """Handle mouse movement event"""
        # Capture data
        data = self.data_capture.capture_mouse_event(event)
        
        # Process (write output, update trail/cursor/display, store position)
        self.tabletOrMouseEvent(data, is_tablet=False)
        

    def mousePressEvent(self, event):
        """Print mouse click position"""
        print(
            f"Mouse click at: X={event.position().x():.1f}, Y={event.position().y():.1f}"
        )
        sys.stdout.flush()

    def keyPressEvent(self, event):
        """Handle keyboard input using command dispatch"""
        key = event.text()
        
        # Look up handler method name from KEY_COMMANDS dictionary
        # Use original key for space, lowercased for letter keys
        lookup_key = key if key == ' ' else key.lower()
        handler_name = self.KEY_COMMANDS.get(lookup_key)
        
        # Execute handler if found
        if handler_name:
            handler = getattr(self, handler_name, None)
            if handler:
                handler()

In [ ]:
# Run Application — Execute the mouseReMoCo Wacom Tablet Test

# Create or retrieve the Qt application singleton
app = QApplication.instance()
if app is None:
    app = QApplication(sys.argv)

# Initialize window orchestrator (creates Configuration internally)
# Handles: screen detection, dimension calculation, configuration lifecycle, window creation
window_setup = WindowSetup(
    app=app,
    tablet_test_class=MainWindow,
)

# Execute complete initialization pipeline
# Detects screens → calculates circle radii → creates window →
# measures frame insets → corrects drawable dimensions → displays window
main_window = window_setup.create_and_display()

# Update configuration MUST be after main_window knows drawable area
window_setup.update_configuration(
    trail_length=2
    * 3.14159
    * window_setup.config.internal_radius  # ~ 1 lap
    # index_of_difficulty=70.0,  # sets circle perimeter accordingly
)

# Launch the Qt event loop
# Blocks until user closes the window; handles all input and rendering
exit_code = app.exec()

In [ ]:
# read and plot the data after application exits
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

%matplotlib widget

# Load CSV (skip first 2 header rows, use row 2 as column names)
df = pd.read_csv("data.csv", skiprows=2)

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Position trace (trajectory)
axes[0, 0].plot(df["mouseX"], df["mouseY"], "b.-", alpha=0.6, linewidth=0.5)
axes[0, 0].set_xlabel("X Position (px)")
axes[0, 0].set_ylabel("Y Position (px)")
axes[0, 0].set_title("Cursor Trajectory")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axis("equal")

# 2. Pressure over time
df["time_elapsed"] = (
    df["event_timestamp"] - df["event_timestamp"].iloc[0]
) / 1000  # seconds
axes[0, 1].plot(df["time_elapsed"], df["pressure"], "r-", linewidth=1)
axes[0, 1].set_xlabel("Time (s)")
axes[0, 1].set_ylabel("Pressure")
axes[0, 1].set_title("Tablet Pressure Over Time")
axes[0, 1].grid(True, alpha=0.3)

# 3. x and y positions over time
axes[1, 0].plot(df["time_elapsed"], df["mouseX"], label="X Position", alpha=0.7)
axes[1, 0].plot(df["time_elapsed"], df["mouseY"], label="Y Position", alpha=0.7)
axes[1, 0].set_xlabel("Time (s)")
axes[1, 0].set_ylabel("Position (px)")
axes[1, 0].set_title("Cursor Position Over Time")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Tilt angles over time
axes[1, 1].plot(df["time_elapsed"], df["tiltX"], label="Tilt X", alpha=0.7)
axes[1, 1].plot(df["time_elapsed"], df["tiltY"], label="Tilt Y", alpha=0.7)
axes[1, 1].set_xlabel("Time (s)")
axes[1, 1].set_ylabel("Tilt Angle (degrees)")
axes[1, 1].set_title("Stylus Tilt Over Time")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# create a histogram of the latencies between event_timestamp and call_time and a boxplot
# of the latencies in a figure with two subplots that are on top of each other and share
# the x axis, with the boxplot on the top ann no box around the histogram
df["latency_ms"] = (
    df["call_time"]
    - df["event_timestamp"]
    - np.min(df["call_time"] - df["event_timestamp"])
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
# Boxplot
ax1.boxplot(df["latency_ms"], vert=False)
ax1.set_title("Latency Boxplot")
ax1.set_yticks([])  # Hide y-axis ticks
# show the mean and median on the boxplot
mean_latency = np.mean(df["latency_ms"])
median_latency = np.median(df["latency_ms"])
ax1.axvline(mean_latency, color="r", label=f"Mean: {mean_latency:.2f} ms")
ax1.axvline(median_latency, color="g", label=f"Median: {median_latency:.2f} ms")
ax1.legend()


# Histogram
ax2.hist(df["latency_ms"], bins=50, color="skyblue", edgecolor="black")
ax2.set_title("Latency Histogram")
ax2.set_xlabel("Latency (ms)")
ax2.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# compute velocity for x y positions using event_timestamp and call_time with gradient
df["v_x_event"] = np.gradient(df["mouseX"]) / np.gradient(df["event_timestamp"] / 1000) # px/s
df["v_y_event"] = np.gradient(df["mouseY"]) / np.gradient(df["event_timestamp"] / 1000) # px/s
df["v_event"] = np.sqrt(df["v_x_event"]**2 + df["v_y_event"]**2)

df["v_x_call"] = np.gradient(df["mouseX"]) / np.gradient(df["call_time"] / 1000) # px/s
df["v_y_call"] = np.gradient(df["mouseY"]) / np.gradient(df["call_time"] / 1000) # px/s
df["v_call"] = np.sqrt(df["v_x_call"]**2 + df["v_y_call"]**2)

# comput acceleration for both velocities
df["a_event"] = np.gradient(df["v_event"]) / np.gradient(df["event_timestamp"] / 1000) # px/s^2
df["a_call"] = np.gradient(df["v_call"]) / np.gradient(df["call_time"] / 1000) # px/s^2

# plot the two velocities on the same graph and the two accelerations below them
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
# Velocities
ax1.plot(df["time_elapsed"], df["v_event"], label="Velocity (event_timestamp)", alpha=0.7)
ax1.plot(df["time_elapsed"], df["v_call"], label="Velocity (call_time)", alpha=0.7)
ax1.set_title("Cursor Velocity Over Time")
ax1.set_ylabel("Velocity (px/s)")
ax1.legend()
ax1.grid(True, alpha=0.3)   
# Accelerations
ax2.plot(df["time_elapsed"], df["a_event"], label="Acceleration (event_timestamp)", alpha=0.7)
ax2.plot(df["time_elapsed"], df["a_call"], label="Acceleration (call_time)", alpha=0.7)
ax2.set_title("Cursor Acceleration Over Time")
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("Acceleration (px/s²)")
ax2.legend()
ax2.grid(True, alpha=0.3)   

plt.tight_layout()
plt.show()

In [ ]:
# get the sampling rate based on event_timestamp and call_time
df["dt_event"] = np.gradient(df["event_timestamp"] / 1000)
df["dt_call"] = np.gradient(df["call_time"] / 1000)
# check for zero dt values and replace them with nan to avoid division by zero
df.loc[df["dt_event"] == 0, "dt_event"] = np.nan
df.loc[df["dt_call"] == 0, "dt_call"] = np.nan
# compute sampling frequency
df["fs_event"] = 1 / df["dt_event"]
df["fs_call"] = 1 / df["dt_call"]

# remove nan values from fs_event and fs_call for boxplot
fs_event_clean = df["fs_event"].dropna()
fs_call_clean = df["fs_call"].dropna()
nb_nan_event = len(df["fs_event"]) - len(fs_event_clean)
nb_nan_call = len(df["fs_call"]) - len(fs_call_clean)

# Calculate statistics
dt_event_clean = df["dt_event"].dropna()
dt_call_clean = df["dt_call"].dropna()

# Calculate recording duration
duration_event = (df["event_timestamp"].iloc[-1] - df["event_timestamp"].iloc[0]) / 1000  # in seconds
duration_call = (df["call_time"].iloc[-1] - df["call_time"].iloc[0]) / 1000  # in seconds
n_samples = len(df)
mean_fs_event = n_samples / duration_event
mean_fs_call = n_samples / duration_call

# Create a single figure with 6 subplots (3 rows, 2 columns)
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

# Row 1: Delta Time Boxplots
axes[0, 0].boxplot(dt_event_clean, vert=False)
axes[0, 0].axvline(dt_event_clean.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {dt_event_clean.mean():.4f}')
axes[0, 0].axvline(dt_event_clean.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {dt_event_clean.median():.4f}')
axes[0, 0].set_title(f"Delta Time (gradient of event_timestamp)\n{n_samples} samples in {duration_event:.3f}s, {mean_fs_event:.3f} Hz")
axes[0, 0].set_xlabel("Delta Time (s)")
axes[0, 0].legend()

axes[0, 1].boxplot(dt_call_clean, vert=False)
axes[0, 1].axvline(dt_call_clean.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {dt_call_clean.mean():.4f}')
axes[0, 1].axvline(dt_call_clean.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {dt_call_clean.median():.4f}')
axes[0, 1].set_title(f"Delta Time (gradient of call_time)\n{n_samples} samples in {duration_call:.3f}s, {mean_fs_call:.3f} Hz")
axes[0, 1].set_xlabel("Delta Time (s)")
axes[0, 1].legend()

# Row 2: Sampling Rate Histograms
axes[1, 0].hist(df["fs_event"], bins=50, color="lightgreen", edgecolor="black")
axes[1, 0].set_title(f"Sampling Rate (gradient of event_timestamp)\nNaNs removed: {nb_nan_event}")
axes[1, 0].set_xlabel("Sampling Rate (Hz)")
axes[1, 0].set_ylabel("Frequency")

axes[1, 1].hist(df["fs_call"], bins=50, color="lightcoral", edgecolor="black")
axes[1, 1].set_title(f"Sampling Rate (gradient of call_time)\nNaNs removed: {nb_nan_call}")
axes[1, 1].set_xlabel("Sampling Rate (Hz)")
axes[1, 1].set_ylabel("Frequency")

# Row 3: Sampling Rate Boxplots
axes[2, 0].boxplot(fs_event_clean, vert=False)
axes[2, 0].axvline(fs_event_clean.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {fs_event_clean.mean():.1f} Hz')
axes[2, 0].axvline(fs_event_clean.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {fs_event_clean.median():.1f} Hz')
axes[2, 0].set_xlabel("Sampling Rate (Hz)")
axes[2, 0].legend()

axes[2, 1].boxplot(fs_call_clean, vert=False)
axes[2, 1].axvline(fs_call_clean.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {fs_call_clean.mean():.1f} Hz')
axes[2, 1].axvline(fs_call_clean.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {fs_call_clean.median():.1f} Hz')
axes[2, 1].set_xlabel("Sampling Rate (Hz)")
axes[2, 1].legend()

# add a suptitle to the figure
if df["pressure"].max() > 0:
    fig.suptitle(f"Detailed Analysis of {n_samples} samples in {duration_event:.3f}s: {mean_fs_event:.6f} Hz, {1000/mean_fs_event:.3f} ms\n Tablet Input Detected\n", fontsize=16)
else:
    fig.suptitle(f"Detailed Analysis of {n_samples} samples in {duration_event:.3f}s: {mean_fs_event:.6f} Hz, {1000/mean_fs_event:.3f} ms\n Mouse Input Detected\n", fontsize=16)

plt.tight_layout()
plt.show()